In [1]:
from pathlib import Path
from typing import Union, Optional, Literal, Tuple, List
import os
import time
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed, Future
from tqdm import tqdm
import tifffile as tiff

def export_hoa_slices_to_tiff_optimized(
    dataset_or_name: Union[str, object],
    output_dir: Union[str, Path],
    *,
    downsample_level: int = 0,
    privatemetadatapath: Optional[Union[str, Path]] = None,
    z_start: Optional[int] = None,
    z_stop: Optional[int] = None,
    filename_prefix: str = "slice",
    reverse: bool = True,
    renumber: bool = False,
    # performance knobs
    chunk_z: int = 128,                       # batch this many z-slices into RAM at once
    workers: Optional[int] = 32,              # writer threads
    compression: Optional[Literal["none","zlib","lzw"]] = "none",
    compress_level: int = 1,                  # zlib level 1 ≈ good speed/size tradeoff
    prefetch: bool = True,                    # overlap reading next chunk with writing current
    bigtiff: Optional[bool] = False           # single-slice files; usually False is fine
) -> None:
    """
    Very fast CPU export of uint16/uint8 z-slices to individual TIFF files.

    Speed features:
      - Chunked contiguous loads across z to reduce backend seeks
      - Optional double-buffer prefetch: read next chunk while writing current
      - ThreadPoolExecutor to parallelize TIFF encoding & file writes
      - Minimal per-slice work (assumes data already uint16/uint8)
      - tifffile backend with controllable compression (none/zlib/lzw)

    Filenames:
      - renumber=False: {prefix}_{z:0{pad}d}.tif
      - renumber=True:  {prefix}_{i:0{pad}d}.tif (continuous across chunks)
    """
    # Lazy import to avoid hard dependency if not needed elsewhere
    import hoa_tools.dataset as hoa_dataset

    # Resolve dataset
    if isinstance(dataset_or_name, str):
        if privatemetadatapath is not None:
            hoa_dataset.change_metadata_directory(Path(privatemetadatapath))
        dataset = hoa_dataset.get_dataset(dataset_or_name)
    else:
        dataset = dataset_or_name

    da = dataset.data_array(downsample_level=downsample_level)
    if "z" not in da.sizes:
        raise ValueError(f"Expected 'z' dimension, got {da.dims}")

    nz = int(da.sizes["z"])
    z0 = 0 if z_start is None else max(0, int(z_start))
    z1 = nz if z_stop is None else min(nz, int(z_stop))
    if not (0 <= z0 < z1 <= nz):
        raise ValueError(f"Invalid z-range: z_start={z0}, z_stop={z1}, total={nz}")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    num = z1 - z0
    pad = max(4, len(str((num - 1) if renumber else (z1 - 1))))
    z_seq: List[int] = list(range(z1 - 1, z0 - 1, -1)) if reverse else list(range(z0, z1))

    # Decide writer args
    if compression is None or compression == "none":
        tiff_args = dict(compression=None)
    else:
        tiff_args = dict(compression=compression)
        if compression == "zlib":
            tiff_args["compressionargs"] = {"level": int(compress_level)}

    # Workers default if None
    if workers is None:
        workers = min(32, max(1, (os.cpu_count() or 8) * 2))

    # Utilities
    def ensure_2d_uint(arr: np.ndarray) -> np.ndarray:
        # expect (y,x) or (y,x,1)
        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = arr[..., 0]
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D slice, got shape {arr.shape}")
        if arr.dtype == np.uint16 or arr.dtype == np.uint8:
            return np.ascontiguousarray(arr)
        # Safety fallback: clip to uint16 (should not trigger for your data)
        return np.ascontiguousarray(np.clip(arr, 0, 65535).astype(np.uint16))

    def out_path_for(idx: int, z: int) -> Path:
        name_idx = idx if renumber else z
        return output_dir / f"{filename_prefix}_{name_idx:0{pad}d}.tif"

    def write_one(path: Path, img: np.ndarray) -> Tuple[Path, int]:
        # return on-disk size to compute actual MB/s (esp. if compressed)
        tiff.imwrite(
            str(path),
            img,
            bigtiff=bool(bigtiff),
            photometric="minisblack",
            metadata=None,  # skip ImageJ/OME metadata for speed
            **tiff_args
        )
        try:
            return path, os.path.getsize(path)
        except Exception:
            return path, img.nbytes  # fallback

    # Build chunk index list
    blocks: List[List[int]] = [z_seq[i:i + chunk_z] for i in range(0, num, chunk_z)]

    # Telemetry
    t0 = time.perf_counter()
    total_bytes_written = 0
    slices_written = 0

    # Estimate bytes/slice for a quick preflight
    # Load a single slice header/content cheaply
    probe_z = z_seq[0]
    probe = np.asarray(da.isel(z=probe_z).values)
    probe = ensure_2d_uint(probe)
    bytes_per_slice_uncompressed = probe.nbytes  # 10k*10k*2 ≈ 200,000,000 bytes

    # Reader that loads a contiguous block [zmin..zmax]
    def load_block(z_block: List[int]) -> Tuple[np.ndarray, int]:
        zmin, zmax = min(z_block), max(z_block)
        blk = da.isel(z=slice(zmin, zmax + 1)).values  # (z, y, x)
        return np.asarray(blk), zmin

    renum_counter = 0

    if prefetch and len(blocks) > 1:
        # Double-buffer: 1 reader thread + N writer threads
        read_pool = ThreadPoolExecutor(max_workers=1)
        next_read: Future = read_pool.submit(load_block, blocks[0])

        with ThreadPoolExecutor(max_workers=workers) as write_pool, \
             tqdm(total=num, desc="Exporting z-slices (prefetch)") as pbar:

            for bi, z_block in enumerate(blocks):
                block, zmin = next_read.result()

                # Kick off reading the next block ASAP
                if bi + 1 < len(blocks):
                    next_read = read_pool.submit(load_block, blocks[bi + 1])

                # Fan out writes for this block
                futures = []
                for z in z_block:
                    sl = block[z - zmin]
                    img = ensure_2d_uint(sl)
                    idx = renum_counter if renumber else z
                    path = out_path_for(idx, z)
                    futures.append(write_pool.submit(write_one, path, img))
                    renum_counter += 1

                for f in as_completed(futures):
                    path, nbytes = f.result()
                    total_bytes_written += nbytes
                    slices_written += 1
                    pbar.update(1)

        read_pool.shutdown(wait=True)

    else:
        # Simple: load block, then write block (still parallel writes)
        with ThreadPoolExecutor(max_workers=workers) as write_pool, \
             tqdm(total=num, desc="Exporting z-slices") as pbar:

            for z_block in blocks:
                block, zmin = load_block(z_block)

                futures = []
                for z in z_block:
                    sl = block[z - zmin]
                    img = ensure_2d_uint(sl)
                    idx = renum_counter if renumber else z
                    path = out_path_for(idx, z)
                    futures.append(write_pool.submit(write_one, path, img))
                    renum_counter += 1

                for f in as_completed(futures):
                    path, nbytes = f.result()
                    total_bytes_written += nbytes
                    slices_written += 1
                    pbar.update(1)

    t1 = time.perf_counter()
    elapsed = max(1e-6, t1 - t0)
    # Two throughput metrics:
    # - Effective MB/s = on-disk bytes / elapsed (reflects compression & FS)
    # - Raw MB/s = uncompressed bytes / elapsed (useful when compression=None)
    MB = 1024 * 1024
    effective_MBps = total_bytes_written / MB / elapsed
    raw_MBps = (slices_written * bytes_per_slice_uncompressed) / MB / elapsed

    print(f"Done. Wrote {slices_written} TIFF slices to: {output_dir}")
    print(f"Elapsed: {elapsed:.2f} s | "
          f"Effective write: {effective_MBps:.1f} MB/s | "
          f"Raw stream: {raw_MBps:.1f} MB/s | "
          f"Avg file size: {total_bytes_written / max(1, slices_written) / MB:.2f} MB")


In [2]:
export_hoa_slices_to_tiff_optimized(
    dataset_or_name="K334_kidney_overview_20.022um_bm18",
    output_dir="D:/ucemlef/kidney/glomeruli/K334_kidney_overview_40.044um_bm18",
    downsample_level=1,  
    privatemetadatapath="E:/thierry/private-hoa-metadata/metadata1",
    reverse=False,
    chunk_z=256,
    workers=16
)



Exporting z-slices (prefetch): 100%|██████████| 3216/3216 [03:06<00:00, 17.28it/s]

Done. Wrote 3216 TIFF slices to: D:\ucemlef\kidney\glomeruli\K334_kidney_overview_40.044um_bm18
Elapsed: 191.59 s | Effective write: 114.8 MB/s | Raw stream: 114.8 MB/s | Avg file size: 6.84 MB
